In [2]:
!pip -q install -U transformers accelerate safetensors

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

print("Qwen3-1.7B loaded!")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-1.7B loaded!


In [4]:
# Freeze all model parameters
model.requires_grad_(False)

# Evaluation mode
model.eval()

# Verify
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 1,720,574,976
Trainable parameters: 0


In [5]:
ordered_text = """
Neural Networks

An artificial neural network consists
of multiple layers of neurons.

ReLU activation function: f(x) = max(0, x).

One problem with ReLU is dying ReLU.

Leaky ReLU allows a small nonzero gradient
for negative inputs.
"""

print(ordered_text)


Neural Networks

An artificial neural network consists
of multiple layers of neurons.

ReLU activation function: f(x) = max(0, x).

One problem with ReLU is dying ReLU.

Leaky ReLU allows a small nonzero gradient
for negative inputs.



In [6]:
def generate_study_guide(ordered_text: str) -> str:

    if not isinstance(ordered_text, str):
        raise TypeError("Input must be a string.")

    if not ordered_text.strip():
        raise ValueError("Input notes cannot be empty.")

    system_prompt = """
You are a study assistant.

Convert OCR-recognized handwritten lecture notes
into a structured Markdown study guide.

Follow these rules:
1. Use only information supported by the input notes.
2. Preserve important terms and definitions.
3. Correct obvious OCR errors only when confident.
4. Do not invent information or equations.
5. If the notes are unclear, acknowledge uncertainty.

Output the following sections:

# Study Guide
## Organized Notes
## Summary
## Review Suggestions
## Self-Test Quiz

Generate 5 quiz questions with answers.
For every question, include supporting evidence
from the original notes.

If there is insufficient information for five
distinct questions, generate fewer questions.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": f"Lecture notes:\n\n{ordered_text}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=900,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [7]:
markdown_output = generate_study_guide(ordered_text)

print(markdown_output)

# Study Guide
## Organized Notes
An artificial neural network consists of multiple layers of neurons.

ReLU activation function: $ f(x) = \max(0, x) $.

One problem with ReLU is dying ReLU.

Leaky ReLU allows a small nonzero gradient for negative inputs.

## Summary
- An artificial neural network consists of multiple layers of neurons.
- The ReLU activation function is defined as $ f(x) = \max(0, x) $.
- One issue with ReLU is the "dying ReLU" problem.
- Leaky ReLU addresses the "dying ReLU" problem by allowing a small nonzero gradient for negative inputs.

## Review Suggestions
- Practice implementing ReLU and Leaky ReLU in code.
- Understand the "dying ReLU" problem and its implications for neural network training.
- Explore how Leaky ReLU can improve model performance compared to ReLU.

## Self-Test Quiz
### Question 1
What is the definition of the ReLU activation function?

**Answer:** The ReLU activation function is defined as $ f(x) = \max(0, x) $.

**Supporting Evidence:** "ReLU

In [8]:
from pathlib import Path

output_file = Path("/content/study_guide.md")

output_file.write_text(
    markdown_output,
    encoding="utf-8"
)

print("Saved:", output_file)

Saved: /content/study_guide.md


In [9]:
from IPython.display import Markdown, display

display(Markdown(markdown_output))

# Study Guide
## Organized Notes
An artificial neural network consists of multiple layers of neurons.

ReLU activation function: $ f(x) = \max(0, x) $.

One problem with ReLU is dying ReLU.

Leaky ReLU allows a small nonzero gradient for negative inputs.

## Summary
- An artificial neural network consists of multiple layers of neurons.
- The ReLU activation function is defined as $ f(x) = \max(0, x) $.
- One issue with ReLU is the "dying ReLU" problem.
- Leaky ReLU addresses the "dying ReLU" problem by allowing a small nonzero gradient for negative inputs.

## Review Suggestions
- Practice implementing ReLU and Leaky ReLU in code.
- Understand the "dying ReLU" problem and its implications for neural network training.
- Explore how Leaky ReLU can improve model performance compared to ReLU.

## Self-Test Quiz
### Question 1
What is the definition of the ReLU activation function?

**Answer:** The ReLU activation function is defined as $ f(x) = \max(0, x) $.

**Supporting Evidence:** "ReLU activation function: $ f(x) = \max(0, x) $."

### Question 2
What is the "dying ReLU" problem?

**Answer:** The "dying ReLU" problem refers to the situation where neurons with ReLU activation function become inactive and no longer contribute to the model's learning.

**Supporting Evidence:** "One problem with ReLU is dying ReLU."

### Question 3
How does Leaky ReLU address the "dying ReLU" problem?

**Answer:** Leaky ReLU allows a small nonzero gradient for negative inputs, which prevents neurons from becoming inactive and becoming "dying."

**Supporting Evidence:** "Leaky ReLU allows a small nonzero gradient for negative inputs."

### Question 4
What is the purpose of the Leaky ReLU activation function?

**Answer:** The purpose of the Leaky ReLU activation function is to address the "dying ReLU" problem by allowing a small nonzero gradient for negative inputs.

**Supporting Evidence:** "Leaky ReLU allows a small nonzero gradient for negative inputs."

### Question 5
What is the difference between ReLU and Leaky ReLU?

**Answer:** ReLU sets the output to zero for negative inputs, while Leaky ReLU allows a small nonzero gradient for negative inputs.

**Supporting Evidence:** "ReLU activation function: $ f(x) = \max(0, x) $." and "Leaky ReLU allows a small nonzero gradient for negative inputs."

test A: wrong input

In [10]:
noisy_notes = """
Neura1 Netw0rks
ReLU = max(0, x)
Problern: dying ReLU
"""

print(generate_study_guide(noisy_notes))

# Study Guide
## Organized Notes
### Neural Networks
- **ReLU** = max(0, x)

### Problem: Dying ReLU
- **ReLU** is a non-linear activation function that outputs the input directly if it's positive, and zero otherwise.
- **Dying ReLU** is a phenomenon where neurons in a network become inactive and always output zero, leading to a loss of information.

## Summary
- **ReLU** is defined as max(0, x).
- **Dying ReLU** occurs when neurons become inactive and always output zero, leading to a loss of information.

## Review Suggestions
- Understand the definition and properties of the ReLU function.
- Recognize the phenomenon of "dying ReLU" and its implications for neural network performance.

## Self-Test Quiz

### Question 1
What is the definition of the ReLU function?

**Answer:**  
The ReLU function is defined as max(0, x).

**Supporting Evidence:**  
"ReLU = max(0, x)" from the lecture notes.

### Question 2
What is the phenomenon called when neurons in a neural network become inactive a

test B: right input

In [11]:
short_notes = "ReLU = max(0, x)"

print(generate_study_guide(short_notes))

# Study Guide
## Organized Notes
ReLU = max(0, x)

## Summary
ReLU is a type of activation function commonly used in neural networks. It outputs the maximum of 0 and the input value. This function introduces non-linearity into the network, allowing it to learn complex patterns.

## Review Suggestions
1. Review the definition of ReLU and its mathematical form.
2. Understand the purpose of non-linear activation functions in neural networks.
3. Practice applying the ReLU function to different input values.

## Self-Test Quiz

### Question 1
What is the mathematical form of the ReLU function?

**Answer:**  
ReLU = max(0, x)

**Supporting Evidence:**  
The lecture notes state: "ReLU = max(0, x)"

### Question 2
What does the ReLU function do to the input values?

**Answer:**  
The ReLU function outputs the maximum of 0 and the input value.

**Supporting Evidence:**  
The lecture notes state: "ReLU = max(0, x)"

### Question 3
Why is the ReLU function used in neural networks?

**Answer:**  
